In [2]:
import av
import numpy as np
from transformers import AutoImageProcessor, TimesformerModel


d:\major project\Video_captioning_code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
np.random.seed(0)


def read_video_pyav(container, indices):
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])


def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    converted_len = int(clip_len * frame_sample_rate)
    end_idx = np.random.randint(converted_len, seg_len)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    return indices

In [ ]:
#  Load your own video
container = av.open("my_video.mp4")

#  Sample 8 frames (with 4-frame interval)
indices = sample_frame_indices(
    clip_len=8,
    frame_sample_rate=4,
    seg_len=container.streams.video[0].frames
)



In [ ]:
# Decode those frames
video = read_video_pyav(container, indices)

#  Preprocess
image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = TimesformerModel.from_pretrained("facebook/timesformer-base-finetuned-k400")
inputs = image_processor(list(video), return_tensors="pt")

#  Inference
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state

#  Show shape
print(list(last_hidden_states.shape))  # ➜ [1, 8, 768]